In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_curve, roc_auc_score, classification_report

plt.rcParams.update({'font.size': 24, 
                     'figure.figsize': (10, 10), 
                     'figure.dpi': 100, 
                     'font.sans-serif': 'Arial',
                     'axes.linewidth': 2.0,
                     'lines.linewidth': 3.0,
                     'xtick.major.width': 2.0,
                     'xtick.minor.visible': True,
                     'xtick.minor.width': 2,
                     'xtick.top': True,
                     'xtick.direction': 'in',
                     'xtick.major.size': 10.0,
                     'xtick.minor.size': 4.0,
                     'xtick.minor.ndivs': 2,
                     'ytick.major.width': 2.0,
                     'ytick.minor.visible': True,
                     'ytick.minor.width': 2,
                     'ytick.right': True,
                     'ytick.direction': 'in',
                     'ytick.major.size': 10.0,
                     'ytick.minor.size': 4.0,
                     'ytick.minor.ndivs': 2
                    })

cwd = os.getcwd()

target = 'is_metal'  # Options: 'is_stable', 'is_metal'
file_path = f"{cwd}\\pkl\\{target}_results"
output_file_path = f"{file_path}\\final"

if target == 'is_stable':
    target_names = ['Unstable', 'Stable']
    ground_truth_name = 'DFT Predicted Stability'
    prediction_name = 'GBFS Predicted Stability'
    stability_filter = False
    metallicity_filter = False
elif target == 'is_metal':
    target_names = ['Metal', 'Non-metal']
    ground_truth_name = 'DFT Predicted Metallicity'
    prediction_name = 'GBFS Predicted Metallicity'
    stability_filter = True
    metallicity_filter = False
elif target == 'is_gap_direct':
    target_names = ['Indirect', 'Direct']
    ground_truth_name = 'DFT Predicted Bandgap Type'
    prediction_name = 'GBFS Predicted Bandgap Type'
    stability_filter = True
    metallicity_filter = True
else:
    raise ValueError("Invalid target specified.")

In [ ]:
df_pred = joblib.load(f"{file_path}\\final\\2d_combined_{target}_pred.pkl")
df_pr = joblib.load(f"{file_path}\\final\\2d_combined_{target}_final_pr.pkl")
df_roc = joblib.load(f"{file_path}\\final\\2d_combined_{target}_final_roc.pkl")
df_data = joblib.load(f"{file_path}\\..\\2d_combined_data.pkl")

df_pred.head()

#filter df_data to only include rows with the same index as the df_pred['task_id'] entry
df_data_filtered = df_data[df_data.index.isin(df_pred['task_id'])]
#reorder df_data_filtered to match the order of df_pred['task_id']
df_data_filtered = df_data_filtered.reindex(df_pred['task_id'])

if stability_filter:
    df_pred = df_pred[df_data_filtered['is_stable'].values]
if metallicity_filter:
    df_pred = df_pred[~df_data_filtered['is_metal'].values]

print(f"Number of samples after filtering: {len(df_pred)}")

#form confusion matrix and calculate precision, recall, F1 score

y_true = df_pred['act_target']
y_pred = df_pred['pred_target']
cm = confusion_matrix(y_true, y_pred)

# calculate overall accuracy as a percentage
acc = (np.trace(cm) / np.sum(cm))*100
tpr = cm[1,1]/np.sum(cm[1,:])  # sensitivity, recall
tnr = cm[0,0]/np.sum(cm[0,:])  # specificity
bal_acc = ((tpr + tnr)/2)*100

precision = precision_score(y_true, y_pred)
precision_weighted = precision_score(y_true, y_pred, average='weighted')
precision_macro = precision_score(y_true, y_pred, average='macro')

recall = recall_score(y_true, y_pred)
recall_weighted = recall_score(y_true, y_pred, average='weighted')
recall_macro = recall_score(y_true, y_pred, average='macro')

f1 = f1_score(y_true, y_pred)
f1_weighted = f1_score(y_true, y_pred, average='weighted')
f1_macro = f1_score(y_true, y_pred, average='macro')

auc = roc_auc_score(y_true, y_pred)

# Print classification report
print(classification_report(y_true, y_pred, target_names=target_names, digits=3))

print(f'Confusion Matrix:\n{cm}')

print(f'Overall Accuracy: {acc:.1f}%')
print(f'Balanced Accuracy: {bal_acc:.1f}%')
print(f'True Positive Rate (Sensitivity): {tpr:.3f}')
print(f'True Negative Rate (Specificity): {tnr:.3f}')

print(f'Precision from predictions: {precision:.3f}')
print(f'Weighted Precision from predictions: {precision_weighted:.3f}')
print(f'Macro Precision from predictions: {precision_macro:.3f}')

print(f'Recall from predictions: {recall:.3f}')
print(f'Weighted Recall from predictions: {recall_weighted:.3f}')
print(f'Macro Recall from predictions: {recall_macro:.3f}')

print(f'F1 Score from predictions: {f1:.3f}')
print(f'Weighted F1 Score from predictions: {f1_weighted:.3f}')
print(f'Macro F1 Score from predictions: {f1_macro:.3f}')

print(f'AUC from predictions: {auc:.3f}')

# Print classification report to a text file
with open(f"{output_file_path}\\2d_combined_{target}_filtered_model_evaluation_results.txt", "w") as f:
    f.write(classification_report(y_true, y_pred, target_names=target_names, digits=3))
    f.write('\n')
    f.write(f'Confusion Matrix:\n{cm}\n')
    f.write(f'Overall Accuracy: {acc:.1f}%')
    f.write(f'Balanced Accuracy: {bal_acc:.1f}%')
    f.write(f'True Positive Rate (Sensitivity): {tpr:.3f}')
    f.write(f'True Negative Rate (Specificity): {tnr:.3f}')
    f.write(f'Precision from predictions: {precision}\n')
    f.write(f'Weighted Precision from predictions: {precision_weighted}\n')
    f.write(f'Macro Precision from predictions: {precision_macro}\n')
    f.write(f'Recall from predictions: {recall}\n')
    f.write(f'Weighted Recall from predictions: {recall_weighted}\n')
    f.write(f'Macro Recall from predictions: {recall_macro}\n')
    f.write(f'F1 Score from predictions: {f1}\n')
    f.write(f'Weighted F1 Score from predictions: {f1_weighted}\n')
    f.write(f'Macro F1 Score from predictions: {f1_macro}\n')
    f.write(f'AUC from predictions: {auc}\n')


In [ ]:
# #plot confusion matrix
# plt.figure(figsize=(8, 8))
# plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)

# # Force perfect squares
# plt.gca().set_aspect('equal')

# tick_marks = np.arange(2)
# plt.xticks(tick_marks, target_names)
# plt.yticks(tick_marks, target_names, rotation=90)
# plt.tick_params(axis='both', which='both', length=0)

# # Remove bounding box around the plot
# for spine in plt.gca().spines.values():
#     spine.set_visible(False)

# thresh = cm.max() / 2.
# for i, j in np.ndindex(cm.shape):
#     plt.text(j, i, format(cm[i, j], 'd'),
#              horizontalalignment="center",
#              color="white" if cm[i, j] > thresh else "black",
#              fontsize=32)
# plt.ylabel(ground_truth_name)
# plt.xlabel(prediction_name)
# plt.tight_layout()
# plt.grid(False)

# # insert F1 score, weighted F1 and macro F1 into confusion matrix plot
# plt.text(
#     1.4, 1.2,  # coordinates just above/below the plot
#     f"$F_1$ Score = {f1:.3f}",
#     ha='right', va='center'
# )
# plt.text(
#     1.4, 1.3,  # coordinates just above/below the plot
#     f"Weighted $F_1$ Score = {f1_weighted:.3f}",
#     ha='right', va='center'
# )
# plt.text(
#     1.4, 1.4,  # coordinates just above/below the plot
#     f"Macro $F_1$ Score = {f1_macro:.3f}",
#     ha='right', va='center'
# )


# plt.savefig(f"{output_file_path}\\2d_combined_{target}_filtered_confusion_matrix.png", dpi=300)
# plt.show()


In [ ]:
#plot precision-recall curve from df_pr dataframe 
plt.figure(figsize=(10, 10))
plt.plot(df_pr['Recall'], df_pr['Precision'], label='Precision-Recall')
plt.plot([0, 1], [0.5, 0.5], label='Baseline', linestyle='--', color='black')
plt.xlim([0.0, 1.0])
plt.ylim([0.00000001, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.tight_layout()
plt.grid()

#add Average Precision, weighted precision and macro precision as text to plot
plt.text(0.98, 0.15, f'Average Precision = {precision:.3f}', fontsize=24, ha='right', va='center')
plt.text(0.98, 0.1, f'Weighted Precision = {precision_weighted:.3f}', fontsize=24, ha='right', va='center')
plt.text(0.98, 0.05, f'Macro Precision = {precision_macro:.3f}', fontsize=24, ha='right', va='center')


plt.savefig(f'{output_file_path}//{target}_filtered_pr.png', dpi=300)
plt.show()

In [ ]:
#plot roc curve from df_roc dataframe 
plt.figure(figsize=(10, 10))
plt.plot(df_roc['FPR'], df_roc['TPR'], label='ROC Curve')
plt.plot([0, 1], [0, 1], label='Baseline', linestyle='--', color='black')
plt.text(0.6, 0.05 , f'AUC-ROC = {auc:.3f}', fontsize=24, ha='right', va='center')
plt.xlim([0.0, 1.0])
plt.ylim([0.00000001, 1.05])
plt.legend()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.tight_layout()
plt.grid()
plt.savefig(f'{output_file_path}//{target}_filtered_roc.png', dpi=300)
plt.show()

In [ ]:
#plot det curve from df_roc dataframe
df_det = pd.DataFrame()

# Convert ROC to DET coordinates
df_det['FPR'] = df_roc['FPR']  # False Positive Rate
df_det['FNR'] = 1 - df_roc['TPR']  # False Negative Rate

# Apply probit (inverse Gaussian CDF) transform
x_det = norm.ppf(df_det['FPR'])
y_det = norm.ppf(df_det['FNR'])

# Plot DET curve
plt.figure(figsize=(10, 10))
plt.plot(x_det, y_det, label='DET Curve')
plt.xlabel("False Positive Rate [%]")
plt.ylabel("False Negative Rate [%]")
plt.legend()
plt.grid(True)

#hide minor ticks
plt.minorticks_off()

# Set reasonable axis ticks (percentages mapped via norm.ppf)
ticks = [0.001, 0.01, 0.05, 0.2, 0.5, 0.8, 0.95, 0.99, 0.999]  #[0.001, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 0.9, 0.95, 0.98, 0.99, 0.999]
tick_labels = [f"{t*100:.1f}" for t in ticks]
plt.xticks(norm.ppf(ticks), tick_labels) #, rotation=90)
plt.yticks(norm.ppf(ticks), tick_labels)

# plt.tight_layout()
plt.tight_layout()
plt.savefig(f'{output_file_path}//{target}_filtered_det.png', dpi=300)
plt.show()




# plt.figure(figsize=(10, 10))
# plt.plot(df_roc['FPR'], df_roc['TPR'], label='Receiver Operating Characteristic')
# plt.plot([0, 1], [0, 1], label='Random Classifier', linestyle='--', color='black')
# plt.xlim([0.0, 1.0])
# plt.ylim([0.00000001, 1.05])
# plt.legend()
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.savefig(f'{output_file_path}//{target}_final_roc.png', dpi=300)

In [ ]:
# plot all of the above performance figures together in a single figure with subplots
fig, axs = plt.subplots(2, 2, figsize=(20, 20))

# (a) Confusion Matrix
ax = axs[0, 0]
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_aspect('equal')
tick_marks = np.arange(2)
ax.set_xticks(tick_marks)
ax.set_xticklabels(target_names)
ax.set_yticks(tick_marks)
ax.set_yticklabels(target_names, rotation=90)
ax.tick_params(axis='both', which='both', length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    ax.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center", color="white" if cm[i, j] > thresh else "black", fontsize=36)
ax.set_ylabel(ground_truth_name)
ax.set_xlabel(prediction_name)
ax.grid(False)
ax.text(-0.4, 1.15, f"Accuracy = {acc:.1f}%", ha='left', va='center')
ax.text(-0.4, 1.25, f"Bal. Acc. = {bal_acc:.1f}%", ha='left', va='center')
ax.text(-0.4, 1.35, f"TPR = {tpr:.3f}", ha='left', va='center')
ax.text(-0.4, 1.45, f"TNR = {tnr:.3f}", ha='left', va='center')
ax.text(1.4, 1.25, f"$F_1$-Score = {f1:.3f}", ha='right', va='center')
ax.text(1.4, 1.35, f"Weighted $F_1$-Score = {f1_weighted:.3f}", ha='right', va='center')
ax.text(1.4, 1.45, f"Macro $F_1$-Score = {f1_macro:.3f}", ha='right', va='center')
ax.set_title('(a)', loc='left', fontsize=28, fontweight='bold', x=-0.12, y=1.05)

# (b) Precision-Recall Curve
ax = axs[0, 1]
ax.plot(df_pr['Recall'], df_pr['Precision'], label='Precision-Recall')
ax.plot([0, 1], [0.5, 0.5], label='Baseline', linestyle='--', color='black')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.00000001, 1.05])
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend()
ax.grid()
ax.text(0.98, 0.15, f'Average Precision = {precision:.3f}', fontsize=24, ha='right', va='center')
ax.text(0.98, 0.1, f'Weighted Precision = {precision_weighted:.3f}', fontsize=24, ha='right', va='center')
ax.text(0.98, 0.05, f'Macro Precision = {precision_macro:.3f}', fontsize=24, ha='right', va='center')
ax.set_title('(b)', loc='left', fontsize=28, fontweight='bold', x=-0.12, y=1.05)

# (c) ROC Curve
ax = axs[1, 0]
ax.plot(df_roc['FPR'], df_roc['TPR'], label='ROC Curve')
ax.plot([0, 1], [0, 1], label='Baseline', linestyle='--', color='black')
ax.text(0.6, 0.05 , f'AUC-ROC = {auc:.3f}', fontsize=24, ha='right', va='center')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.00000001, 1.05])
ax.legend()
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.grid()
ax.set_title('(c)', loc='left', fontsize=28, fontweight='bold', x=-0.12, y=1.05)

# (d) DET Curve
ax = axs[1, 1]
df_det = pd.DataFrame()
df_det['FPR'] = df_roc['FPR']
df_det['FNR'] = 1 - df_roc['TPR']
x_det = norm.ppf(df_det['FPR'])
y_det = norm.ppf(df_det['FNR'])
ax.plot(x_det, y_det, label='DET Curve')
ax.set_xlabel("False Positive Rate [%]")
ax.set_ylabel("False Negative Rate [%]")
ax.legend()
ax.grid(True)
ax.minorticks_off()
ticks = [0.001, 0.01, 0.05, 0.2, 0.5, 0.8, 0.95, 0.99, 0.999]
tick_labels = [f"{t*100:.1f}" for t in ticks]
ax.set_xticks(norm.ppf(ticks))
ax.set_xticklabels(tick_labels)
ax.set_yticks(norm.ppf(ticks))
ax.set_yticklabels(tick_labels)
ax.set_title('(d)', loc='left', fontsize=28, fontweight='bold', x=-0.12, y=1.05)

plt.tight_layout()
plt.savefig(f'{output_file_path}//{target}_filtered_all_performance.png', dpi=300)
plt.savefig(f'{output_file_path}//{target}_filtered_all_performance.pdf', dpi=300)
plt.show()